In [ ]:
import pandas as pd

In [ ]:
# Read Excel
file_path = r"C:\Users\0193284\Downloads\Week 12.xlsx"

df = pd.read_excel(file_path)

# Convert datetime
df["TimeDate"] = pd.to_datetime(df["TimeDate"])

# Sort by Id and Time
df = df.sort_values(["IdNumber", "TimeDate"])

results = []

for id_num, group in df.groupby("IdNumber"):
    arrive_time = None

    for _, row in group.iterrows():

        if row["Direction"] == "Arrive":
            arrive_time = row["TimeDate"]

        elif row["Direction"] == "Depart" and arrive_time is not None:

            minutes = (
                row["TimeDate"] - arrive_time
            ).total_seconds() / 60

            results.append({
                "IdNumber": id_num,
                "ArriveTime": arrive_time,
                "DepartTime": row["TimeDate"],
                "MinutesInTerminal": round(minutes, 2)
            })

            arrive_time = None

result_df = pd.DataFrame(results)

print(result_df)

# Save results
result_df.to_excel("Terminal_Turnaround.xlsx", index=False)

In [ ]:
# Total minutes
total_minutes = result_df["MinutesInTerminal"].sum()

# Count of records
count_minutes = result_df["MinutesInTerminal"].count()

# TTT calculation
TTT = total_minutes / count_minutes if count_minutes != 0 else 0

print("Total Minutes In Terminal:", total_minutes)
print("Count:", count_minutes)
print("TTT (Average Turnaround Time):", TTT)

In [ ]:
fact_terminal = result_df.copy()

fact_terminal["Date"] = fact_terminal["ArriveTime"].dt.date
fact_terminal["Hour"] = fact_terminal["ArriveTime"].dt.hour

# Optional: add shift classification
def get_shift(hour):
    if 6 <= hour < 18:
        return "Day Shift"
    else:
        return "Night Shift"

fact_terminal["Shift"] = fact_terminal["Hour"].apply(get_shift)

# Save for Power BI
fact_terminal.to_excel("Fact_TerminalTurnaround.xlsx", index=False)

In [ ]:
summary_terminal = fact_terminal.groupby("IdNumber").agg(
    TotalMinutes=("MinutesInTerminal", "sum"),
    TripCount=("MinutesInTerminal", "count"),
    AvgMinutes=("MinutesInTerminal", "mean")
).reset_index()

summary_terminal["TTT"] = summary_terminal["TotalMinutes"] / summary_terminal["TripCount"]

summary_terminal.to_excel("Summary_TerminalTurnaround.xlsx", index=False)